# 집 수리 요청 프로세스 분석: SQL 재검증

Python으로 진행한 집 수리 요청 로그 분석(수리 요청 1,000건, 기록 13,262줄)을 **SQL로 다시 계산해 결과를 검증**한 노트북입니다.

| 질문 | SQL로 확인한 것 |
|---|---|
| 1. 단계 사이 시간은 어디서 가장 오래 걸리는가 | 고객 안내 → 현장 점검 대기가 가장 김 (중앙값 49.2시간) |
| 2. 케이스는 어떤 경로로 진행되는가 | 이상 날짜 케이스를 빼면 짧은 경로가 과하게 남는 치우침 발견 |
| 3. 수리 유형별 소요시간은 얼마나 다른가 | B형 0.47일 · P형 2.83일 · E형 6.22일, Python 결과와 일치 |

- 도구: DuckDB (Colab에서 CSV를 바로 SQL로 조회)
- 데이터: 네덜란드 임대주택 기관의 집 수리 요청 이벤트 로그 (`repair2.csv`)

## 0. 준비

`repair2.csv`를 노트북과 같은 폴더(Colab 기본 폴더 `/content`)에 올린 뒤 실행합니다.

In [1]:
!pip install -q duckdb
import duckdb
import pandas as pd
pd.set_option("display.max_colwidth", None)   # 경로 문자열을 자르지 않고 표시

## 1. 데이터 확인

이상 날짜 확인: 정상 기록은 1970년 1월에 몰려 있고, 2020~2022년과 1905년 값이 섞여 있습니다.

In [2]:
duckdb.sql("""
SELECT COUNT(*)                                   AS rows,
       COUNT(DISTINCT caseID)                     AS cases,
       COUNT(*) FILTER (WHERE year(date) <> 1970) AS bad_date_rows,
       ROUND(100.0 * COUNT(*) FILTER (WHERE year(date) <> 1970) / COUNT(*), 1) AS bad_date_pct
FROM 'repair2.csv'
""").df()

,rows,cases,bad_date_rows,bad_date_pct
0,13262,1000,866,6.5


업무별 시작/완료 기록 수. 시작 기록은 일부 업무에만 있어서, **단계 사이 시간은 모든 업무에 있는 완료 기록을 기준으로** 계산합니다. (외부 수리는 완료 기록이 없어 시간 계산에서 제외)

In [3]:
duckdb.sql("""
SELECT taskID,
       COUNT(*) FILTER (WHERE eventtype = 'start')    AS start_rows,
       COUNT(*) FILTER (WHERE eventtype = 'complete') AS complete_rows
FROM 'repair2.csv'
WHERE taskID IS NOT NULL
GROUP BY taskID
ORDER BY complete_rows DESC
""").df()

,taskID,start_rows,complete_rows
0,FirstContact,0,999
1,ArrangeSurvey,926,927
2,ReadyInformClient,0,927
3,SendTicketToFinAdmin,0,927
4,MakeTicket,927,927
5,RepairReady,0,927
6,Survey,927,926
7,TicketReady,0,926
8,InformClientSurvey,0,925
9,InternRepair,547,547


## 2. 단계 사이 시간: 병목은 어디인가

### 처음 발견한 문제: 같은 시각에 끝난 업무

같은 케이스 안에서 완료 시각이 분 단위까지 같은 기록이 많습니다. 시각만으로 정렬하면 둘 중 무엇이 먼저인지 정해지지 않아, **실행할 때마다 결과가 달라졌습니다.**

In [4]:
duckdb.sql("""
WITH ev AS (
  SELECT caseID, taskID, date + time AS ts
  FROM 'repair2.csv'
  WHERE eventtype = 'complete' AND year(date) = 1970
)
SELECT a.taskID AS task_a, b.taskID AS task_b, COUNT(*) AS same_time_cases
FROM ev a
JOIN ev b ON a.caseID = b.caseID AND a.ts = b.ts AND a.taskID < b.taskID
GROUP BY ALL
ORDER BY same_time_cases DESC
""").df()

,task_a,task_b,same_time_cases
0,ArrangeSurvey,InformClientSurvey,805
1,SendTicketToFinAdmin,TicketReady,418
2,ReadyInformClient,TicketReady,405
3,InternRepair,RepairReady,403
4,ImmediateRepair,RepairReady,285
5,FirstContact,InformClientWrongPlace,63
6,ReadyInformClient,SendTicketToFinAdmin,1


### 해결: 시각이 같으면 표준 업무 순서로 정렬

`ORDER BY ts, step_order`로 정렬 기준을 하나 더 두어 결과를 고정했습니다.

In [5]:
duckdb.sql("""
WITH ev AS (
  SELECT caseID, taskID, date + time AS ts,
         CASE taskID
           WHEN 'FirstContact'           THEN 1
           WHEN 'InformClientWrongPlace' THEN 2
           WHEN 'MakeTicket'             THEN 2
           WHEN 'ArrangeSurvey'          THEN 3
           WHEN 'InformClientSurvey'     THEN 4
           WHEN 'Survey'                 THEN 5
           WHEN 'RepairReady'            THEN 7
           WHEN 'ReadyInformClient'      THEN 8
           WHEN 'SendTicketToFinAdmin'   THEN 9
           WHEN 'TicketReady'            THEN 10
           ELSE 6                        -- 수리 3종
         END AS step_order
  FROM 'repair2.csv'
  WHERE eventtype = 'complete'
    AND year(date) = 1970                -- 이상 날짜 기록 제외
),
seq AS (
  SELECT caseID, taskID, ts,
         LAG(taskID) OVER w AS prev_task,
         LAG(ts)     OVER w AS prev_ts
  FROM ev
  WINDOW w AS (PARTITION BY caseID ORDER BY ts, step_order)
)
SELECT prev_task || ' -> ' || taskID AS step,
       COUNT(*) AS cases,
       ROUND(MEDIAN(DATE_DIFF('minute', prev_ts, ts)) / 60, 1) AS median_hours
FROM seq
WHERE prev_task IS NOT NULL
GROUP BY step
HAVING COUNT(*) >= 50
ORDER BY median_hours DESC
""").df()

,step,cases,median_hours
0,ArrangeSurvey -> Survey,55,54.6
1,InformClientSurvey -> Survey,812,49.2
2,Survey -> InternRepair,424,13.4
3,Survey -> RepairReady,94,7.2
4,ReadyInformClient -> SendTicketToFinAdmin,419,2.6
5,RepairReady -> SendTicketToFinAdmin,416,2.5
6,RepairReady -> ReadyInformClient,435,2.5
7,SendTicketToFinAdmin -> ReadyInformClient,398,2.4
8,Survey -> ImmediateRepair,339,2.0
9,FirstContact -> ArrangeSurvey,68,0.2


**결과**: 고객 안내 → 현장 점검이 812건, 중앙값 **49.2시간**으로 가장 깁니다. 점검 → 내부 수리는 13.4시간. Python 분석(49시간, 13시간)과 같은 결론입니다.

`ArrangeSurvey -> Survey` 55건은 중간의 고객 안내 기록이 이상 날짜라 빠진 케이스로, 실제로는 같은 점검 대기 구간입니다.

## 3. 경로: 케이스는 어떤 순서로 진행되는가

케이스별 업무 순서를 `STRING_AGG`로 한 줄로 이어 붙이고, 같은 순서끼리 묶어 셉니다. 마무리 3종은 `Closing` 하나로 합쳤습니다. 여기서는 **날짜가 모두 정상인 케이스만** 사용했습니다.

In [6]:
duckdb.sql("""
WITH ev AS (
  SELECT caseID,
         CASE WHEN taskID IN ('ReadyInformClient', 'SendTicketToFinAdmin', 'TicketReady')
              THEN 'Closing' ELSE taskID END AS task,
         date + time AS ts,
         year(date) = 1970 AS ok_date,
         CASE taskID
           WHEN 'FirstContact'           THEN 1
           WHEN 'InformClientWrongPlace' THEN 2
           WHEN 'MakeTicket'             THEN 2
           WHEN 'ArrangeSurvey'          THEN 3
           WHEN 'InformClientSurvey'     THEN 4
           WHEN 'Survey'                 THEN 5
           WHEN 'RepairReady'            THEN 7
           WHEN 'ReadyInformClient'      THEN 8
           WHEN 'SendTicketToFinAdmin'   THEN 8
           WHEN 'TicketReady'            THEN 8
           ELSE 6
         END AS step_order
  FROM 'repair2.csv'
  WHERE eventtype = 'complete' AND taskID IS NOT NULL
),
clean_cases AS (                -- 날짜가 모두 정상인 케이스만
  SELECT caseID FROM ev GROUP BY caseID HAVING BOOL_AND(ok_date)
),
ordered AS (
  SELECT caseID, task, ts, step_order,
         LAG(task) OVER (PARTITION BY caseID ORDER BY ts, step_order) AS prev_task
  FROM ev
  WHERE caseID IN (SELECT caseID FROM clean_cases)
),
paths AS (
  SELECT caseID, STRING_AGG(task, ' > ' ORDER BY ts, step_order) AS path
  FROM ordered
  WHERE NOT (task = 'Closing' AND COALESCE(prev_task, '') = 'Closing')   -- Closing 중복 제거
  GROUP BY caseID
)
SELECT path,
       COUNT(*) AS cases,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM paths
GROUP BY path
ORDER BY cases DESC
""").df()

,path,cases,pct
0,FirstContact > MakeTicket > ArrangeSurvey > InformClientSurvey > Survey > InternRepair > RepairReady > Closing,222,42.8
1,FirstContact > MakeTicket > ArrangeSurvey > InformClientSurvey > Survey > ImmediateRepair > RepairReady > Closing,172,33.1
2,FirstContact > InformClientWrongPlace,63,12.1
3,FirstContact > MakeTicket > ArrangeSurvey > InformClientSurvey > Survey > RepairReady > Closing,32,6.2
4,FirstContact > MakeTicket > ArrangeSurvey > InformClientSurvey > Survey > InternRepair > InternRepair > RepairReady > Closing,15,2.9
5,FirstContact > MakeTicket > ArrangeSurvey > InformClientSurvey > Survey > ImmediateRepair > InternRepair > RepairReady > Closing,9,1.7
6,FirstContact > MakeTicket > ArrangeSurvey > Survey > ImmediateRepair > RepairReady > Closing,1,0.2
7,MakeTicket > ArrangeSurvey > InformClientSurvey > Survey > ImmediateRepair > RepairReady > Closing > FirstContact,1,0.2
8,FirstContact > MakeTicket > ArrangeSurvey > InformClientSurvey > Survey > ImmediateRepair > InternRepair > InternRepair > RepairReady > Closing,1,0.2
9,FirstContact > MakeTicket > ArrangeSurvey > Survey > InternRepair > RepairReady > Closing,1,0.2


### 발견: 정상 케이스만 남기면 짧은 경로가 과하게 남는다

| 경로 | Python (1,000건, 순서 복원) | SQL (519건, 정상 케이스만) |
|---|---|---|
| 내부 수리 | 41.7% | 42.8% |
| 즉시 수리 | 32.2% | 33.1% |
| 잘못된 장소 | 7.3% | **12.1%** |

잘못된 장소 비율만 크게 늘었습니다. 이유를 확인하기 위해 경로 유형별로 '날짜가 모두 정상인 케이스'가 얼마나 남는지 계산했습니다.

In [7]:
duckdb.sql("""
WITH c AS (
  SELECT caseID,
         COUNT(*) AS complete_rows,
         BOOL_AND(year(date) = 1970) AS clean,
         CASE WHEN BOOL_OR(taskID = 'InformClientWrongPlace') THEN '잘못된 장소'
              WHEN BOOL_OR(taskID = 'InternRepair')           THEN '내부 수리'
              WHEN BOOL_OR(taskID = 'ImmediateRepair')        THEN '즉시 수리'
              ELSE '외부 수리' END AS case_type
  FROM 'repair2.csv'
  WHERE eventtype = 'complete' AND taskID IS NOT NULL
  GROUP BY caseID
)
SELECT case_type,
       COUNT(*)                         AS cases,
       ROUND(AVG(complete_rows), 1)     AS avg_rows,
       SUM(clean::INT)                  AS clean_cases,
       ROUND(100 * AVG(clean::INT), 1)  AS survive_pct,
       ROUND(100 * POWER(0.935, AVG(complete_rows)), 1) AS expected_pct   -- 줄마다 6.5% 확률로 이상 날짜
FROM c
GROUP BY case_type
ORDER BY cases DESC
""").df()

,case_type,cases,avg_rows,clean_cases,survive_pct,expected_pct
0,내부 수리,508,10.1,249.0,49.0,50.7
1,즉시 수리,360,10.0,175.0,48.6,51.1
2,잘못된 장소,73,2.0,63.0,86.3,87.4
3,외부 수리,59,9.0,32.0,54.2,54.6


**해석**: 이상 날짜는 줄마다 약 6.5% 확률로 고르게 섞여 있습니다. 기록이 2줄뿐인 잘못된 장소 케이스는 약 87%가 살아남지만, 약 10줄인 수리 케이스는 절반 정도만 남습니다. 계산값(`expected_pct`)과 실제(`survive_pct`)가 거의 같습니다.

따라서 이상 데이터가 있는 케이스를 통째로 빼면 결과가 짧은 케이스 쪽으로 치우칩니다. Python 분석에서 케이스를 버리지 않고 488건의 순서를 복원한 방식이 더 적절했다는 근거입니다.

## 4. 수리 유형별 소요시간

3번의 결과를 반영해 **케이스는 빼지 않고 이상 날짜인 줄만 제외**했습니다. 소요시간은 정상 날짜 기록 중 가장 이른 시각부터 가장 늦은 시각까지입니다. 수리 유형(`RepairType`)은 티켓 작성 행에만 값이 있어 케이스별로 끌어왔습니다.

In [8]:
duckdb.sql("""
WITH repair_type AS (
  SELECT caseID, MAX(RepairType) AS rtype
  FROM 'repair2.csv'
  GROUP BY caseID
),
duration AS (
  SELECT caseID,
         DATE_DIFF('minute', MIN(date + time), MAX(date + time)) / 60.0 / 24 AS days
  FROM 'repair2.csv'
  WHERE year(date) = 1970
  GROUP BY caseID
)
SELECT COALESCE(rtype, '(잘못된 장소)') AS repair_type,
       COUNT(*) AS cases,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
       ROUND(MEDIAN(days), 2) AS median_days,
       ROUND(AVG(days), 2)    AS avg_days,
       ROUND(MAX(days), 2)    AS max_days
FROM duration
JOIN repair_type USING (caseID)
GROUP BY rtype
ORDER BY median_days
""").df()

,repair_type,cases,pct,median_days,avg_days,max_days
0,(잘못된 장소),73,7.3,0.00,0.00,0.00
1,B,118,11.8,0.47,0.52,1.33
2,P,439,43.9,2.83,3.60,12.27
3,E,370,37.0,6.22,6.63,17.07


**결과**: 중앙값 B형 0.47일, P형 2.83일, E형 6.22일로 Python 분석과 정확히 같습니다. 평균이 중앙값보다 큰 것은 일부 케이스가 매우 오래 걸리기 때문이며, 그래서 중앙값을 기준으로 비교했습니다.

## 5. 정리

| 항목 | Python | SQL | 비고 |
|---|---|---|---|
| 점검 대기 중앙값 | 49시간 | 49.2시간 | 같은 시각 기록의 순서를 고정한 뒤 일치 |
| 점검 → 내부 수리 | 13시간 | 13.4시간 | |
| 수리 유형별 중앙값 | 0.47 / 2.83 / 6.22일 | 0.47 / 2.83 / 6.22일 | 정확히 일치 |
| 잘못된 장소 비율 | 7.3% | 12.1% | 정상 케이스만 쓰면 생기는 치우침 |

**한계**
- 이상 날짜 줄을 빼고 계산해, 처음이나 마지막 기록이 이상 날짜인 케이스는 소요시간이 실제보다 짧게 잡혔을 수 있음
- 외부 수리는 완료 기록이 없어 시간 계산에서 제외
- 1970년 날짜는 실제 날짜가 아닌 상대 시각으로 보고 사용